# Transformaciones y remodelado de DataFrames

In [ ]:
import pandas as pd

# Dataset de ejemplo para las demostraciones
df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'group': ['A', 'A', 'B', 'B', 'C', 'C'],
    'math': [85, 90, 78, 92, 88, 76],
    'physics': [80, 85, 82, 88, 90, 70],
    'chemistry': [75, 88, 80, 95, 85, 72]
})
df

,student,group,math,physics,chemistry
0,Alice,A,85,80,75
1,Bob,A,90,85,88
2,Charlie,B,78,82,80
3,Diana,B,92,88,95
4,Eve,C,88,90,85
5,Frank,C,76,70,72


## Operaciones elemento a elemento: `map`, `apply`, `transform`

### `map`: solo Series, elemento a elemento

`map` se usa exclusivamente sobre **Series** para aplicar una función, un diccionario u otra Series elemento a elemento.

**Casos de uso:**
- Sustituir valores usando un diccionario o una función
- Transformaciones sencillas elemento a elemento sobre una sola columna

In [ ]:
# map con una función
df['math'].map(lambda x: x + 5)  # Suma 5 a cada nota de matemáticas

0    90
1    95
2    83
3    97
4    93
5    81
Name: math, dtype: int64

In [ ]:
# map con un diccionario: útil para sustituciones de categorías
group_names = {'A': 'Alpha', 'B': 'Beta', 'C': 'Gamma'}
df['group'].map(group_names)

0    Alpha
1    Alpha
2     Beta
3     Beta
4    Gamma
5    Gamma
Name: group, dtype: str

In [ ]:
# map devuelve NaN para los valores que no están en el diccionario
incomplete_map = {'A': 'Alpha', 'B': 'Beta'}  # Falta 'C'
df['group'].map(incomplete_map)  # 'C' se convierte en NaN

0    Alpha
1    Alpha
2     Beta
3     Beta
4      NaN
5      NaN
Name: group, dtype: str

> **Nota:** para mapear con un diccionario conservando los valores originales de las claves que falten, usa `replace()` en lugar de `map()`.

In [ ]:
# replace conserva los valores originales de las claves que faltan
df['group'].replace({'A': 'Alpha', 'B': 'Beta'})  # 'C' se queda como 'C'

0    Alpha
1    Alpha
2     Beta
3     Beta
4        C
5        C
Name: group, dtype: str

### `apply`: flexible, funciona con Series y DataFrames

`apply` es más versátil que `map`:
- Sobre una **Series**: aplica una función elemento a elemento (parecido a `map`)
- Sobre un **DataFrame**: aplica una función a lo largo de un eje (filas o columnas)

In [ ]:
# apply sobre una Series: parecido a map
df['math'].apply(lambda x: x + 5)

0    90
1    95
2    83
3    97
4    93
5    81
Name: math, dtype: int64

In [ ]:
# apply sobre un DataFrame: opera sobre columnas enteras (axis=0, por defecto)
df[['math', 'physics', 'chemistry']].apply(lambda col: col.max() - col.min())

math         16
physics      20
chemistry    23
dtype: int64

In [ ]:
# apply sobre un DataFrame: opera sobre filas enteras (axis=1)
df[['math', 'physics', 'chemistry']].apply(lambda row: row.mean(), axis=1)

0    80.000000
1    87.666667
2    80.000000
3    91.666667
4    87.666667
5    72.666667
dtype: float64

**Idea clave:** al usar `apply` sobre un DataFrame, la función recibe una Series entera (una columna o una fila), no valores sueltos.

In [ ]:
# Mostramos qué recibe apply
def show_input(x):
    print(f"Type: {type(x).__name__}, Shape: {x.shape}")
    print(x)
    print("---")
    return x.sum()

df[['math', 'physics']].apply(show_input)  # Cada columna se pasa como una Series

Type: Series, Shape: (6,)
0    85
1    90
2    78
3    92
4    88
5    76
Name: math, dtype: int64
---
Type: Series, Shape: (6,)
0    80
1    85
2    82
3    88
4    90
5    70
Name: physics, dtype: int64
---


math       509
physics    495
dtype: int64

### `transform`: salida con la misma forma, compatible con grupos

`transform` siempre devuelve un resultado con la **misma forma** que la entrada. Esto es crucial para:
- Llevar los cálculos a nivel de grupo de vuelta a cada fila (*broadcasting*)
- Garantizar la alineación con el DataFrame original

In [ ]:
# transform sobre una Series: elemento a elemento, igual que map/apply
df['math'].transform(lambda x: x + 5)

0    90
1    95
2    83
3    97
4    93
5    81
Name: math, dtype: int64

In [ ]:
# transform sobre las columnas de un DataFrame
df[['math', 'physics', 'chemistry']].transform(lambda x: (x - x.mean()) / x.std())

,math,physics,chemistry
0,0.025545,-0.349727,-0.877208
1,0.791910,0.349727,0.643286
2,-1.047364,-0.069945,-0.292403
3,1.098455,0.769400,1.462013
4,0.485364,1.049182,0.292403
5,-1.353910,-1.748637,-1.228091


### Comparación: cuándo usar cada uno

| Método | Funciona con | Entrada a la función | Forma de la salida | Ideal para |
|--------|----------|-------------------|--------------|----------|
| `map` | Solo Series | Un valor | Igual que la entrada | Sustituir valores, transformaciones sencillas |
| `apply` | Series y DataFrame | Valor (Series) o Series (DataFrame) | Puede cambiar | Operaciones flexibles, agregaciones |
| `transform` | Series y DataFrame | Series | **Debe coincidir con la entrada** | Normalización, *broadcasting* por grupos |

In [ ]:
# Operaciones equivalentes: todas suman 5 a las notas de matemáticas
print("map:      ", df['math'].map(lambda x: x + 5).tolist())
print("apply:    ", df['math'].apply(lambda x: x + 5).tolist())
print("transform:", df['math'].transform(lambda x: x + 5).tolist())

map:       [90, 95, 83, 97, 93, 81]
apply:     [90, 95, 83, 97, 93, 81]
transform: [90, 95, 83, 97, 93, 81]


In [ ]:
# Solo map funciona directamente con diccionarios
df['group'].map({'A': 'Alpha', 'B': 'Beta', 'C': 'Gamma'})
# df['group'].apply({'A': 'Alpha'})  # Daría un error
# df['group'].transform({'A': 'Alpha'})  # Daría un error

0    Alpha
1    Alpha
2     Beta
3     Beta
4    Gamma
5    Gamma
Name: group, dtype: str

In [ ]:
# Solo apply puede reducir dimensiones (devolver otra forma)
df[['math', 'physics', 'chemistry']].apply(lambda x: x.sum())  # Devuelve una Series con 3 valores
# df[['math', 'physics', 'chemistry']].transform(lambda x: x.sum())  # daría un ValueError: transform debe mantener la forma

math         509
physics      495
chemistry    495
dtype: int64

## Operaciones de agrupación

### `groupby`: dividir, aplicar, combinar

La operación `groupby` sigue un patrón de **dividir-aplicar-combinar** (*split-apply-combine*):
1. **Dividir** los datos en grupos según una o más claves
2. **Aplicar** una función a cada grupo de forma independiente
3. **Combinar** los resultados en una estructura de datos nueva

In [ ]:
# Creamos un objeto groupby
grouped = df.groupby('group')
print(type(grouped))
print(f"Number of groups: {grouped.ngroups}")
print(f"Groups: {grouped.groups}")

<class 'pandas.api.typing.DataFrameGroupBy'>
Number of groups: 3
Groups: {'A': [0, 1], 'B': [2, 3], 'C': [4, 5]}


In [ ]:
# Recorremos los grupos
for name, group_df in grouped:
    print(f"Group: {name}")
    print(group_df)
    print()

Group: A
  student group  math  physics  chemistry
0   Alice     A    85       80         75
1     Bob     A    90       85         88

Group: B
   student group  math  physics  chemistry
2  Charlie     B    78       82         80
3    Diana     B    92       88         95

Group: C
  student group  math  physics  chemistry
4     Eve     C    88       90         85
5   Frank     C    76       70         72



In [ ]:
# Obtenemos un grupo concreto
grouped.get_group('A')

,student,group,math,physics,chemistry
0,Alice,A,85,80,75
1,Bob,A,90,85,88


In [ ]:
# Agregación básica
df.groupby('group')['math'].mean()

group
A    87.5
B    85.0
C    82.0
Name: math, dtype: float64

In [ ]:
# Varias columnas
df.groupby('group')[['math', 'physics', 'chemistry']].mean()

,math,physics,chemistry
group,,,
A,87.5,82.5,81.5
B,85.0,85.0,87.5
C,82.0,80.0,78.5


### `agg`: varias agregaciones

El método `agg` (o `aggregate`) permite aplicar varias funciones de agregación a la vez, e incluso funciones distintas a columnas distintas.

In [ ]:
# Una sola agregación (equivale a .mean())
df.groupby('group')['math'].agg('mean')

group
A    87.5
B    85.0
C    82.0
Name: math, dtype: float64

In [ ]:
# Varias agregaciones sobre una columna
df.groupby('group')['math'].agg(['mean', 'std', 'min', 'max'])

,mean,std,min,max
group,,,,
A,87.5,3.535534,85,90
B,85.0,9.899495,78,92
C,82.0,8.485281,76,88


In [ ]:
# Varias agregaciones sobre varias columnas
df.groupby('group')[['math', 'physics']].agg(['mean', 'std'])

math           physics           
       mean       std    mean        std
group                                   
A      87.5  3.535534    82.5   3.535534
B      85.0  9.899495    85.0   4.242641
C      82.0  8.485281    80.0  14.142136

In [ ]:
# Agregaciones distintas para columnas distintas usando un diccionario
df.groupby('group').agg({
    'math': 'mean',
    'physics': ['min', 'max'],
    'chemistry': 'std'
})

math physics      chemistry
       mean     min max        std
group                             
A      87.5      80  85   9.192388
B      85.0      82  88  10.606602
C      82.0      70  90   9.192388

In [ ]:
# Agregaciones con nombre (nombres de columna más limpios)
df.groupby('group').agg(
    math_avg=('math', 'mean'),
    physics_range=('physics', lambda x: x.max() - x.min()),
    chemistry_std=('chemistry', 'std')
)

,math_avg,physics_range,chemistry_std
group,,,
A,87.5,5,9.192388
B,85.0,6,10.606602
C,82.0,20,9.192388


### `transform` con `groupby`: llevar los resultados de grupo a cada fila

La verdadera potencia de `transform` brilla al combinarlo con `groupby`. Lleva los cálculos a nivel de grupo de vuelta a cada fila, manteniendo la forma del DataFrame original.

In [ ]:
# Problema: añadir una columna con la media del grupo de cada alumno
# Con apply: devuelve el resultado agregado (3 filas)
df.groupby('group')['math'].apply(lambda x: x.mean())

group
A    87.5
B    85.0
C    82.0
Name: math, dtype: float64

In [ ]:
# Con transform: vuelve a la forma original (6 filas)
df.groupby('group')['math'].transform('mean')

0    87.5
1    87.5
2    85.0
3    85.0
4    82.0
5    82.0
Name: math, dtype: float64

In [ ]:
# Añadimos la media del grupo como columna nueva
df_with_group_mean = df.assign(
    group_math_avg=df.groupby('group')['math'].transform('mean')
)
df_with_group_mean

,student,group,math,physics,chemistry,group_math_avg
0,Alice,A,85,80,75,87.5
1,Bob,A,90,85,88,87.5
2,Charlie,B,78,82,80,85.0
3,Diana,B,92,88,95,85.0
4,Eve,C,88,90,85,82.0
5,Frank,C,76,70,72,82.0


In [ ]:
# Normalizamos dentro de cada grupo (z-score por grupo)
df.assign(
    math_normalized=df.groupby('group')['math'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
)

,student,group,math,physics,chemistry,math_normalized
0,Alice,A,85,80,75,-0.707107
1,Bob,A,90,85,88,0.707107
2,Charlie,B,78,82,80,-0.707107
3,Diana,B,92,88,95,0.707107
4,Eve,C,88,90,85,0.707107
5,Frank,C,76,70,72,-0.707107


In [ ]:
# Ranking dentro de cada grupo
df.assign(
    math_rank_in_group=df.groupby('group')['math'].transform('rank', ascending=False)
)

,student,group,math,physics,chemistry,math_rank_in_group
0,Alice,A,85,80,75,2.0
1,Bob,A,90,85,88,1.0
2,Charlie,B,78,82,80,2.0
3,Diana,B,92,88,95,1.0
4,Eve,C,88,90,85,1.0
5,Frank,C,76,70,72,2.0


### `apply` con `groupby`: operaciones de grupo flexibles

`apply` sobre un objeto agrupado es la opción más flexible. La función recibe el grupo entero como DataFrame y puede devolver:
- Un escalar (agregación)
- Una Series (transformación)
- Un DataFrame (transformaciones complejas)

In [ ]:
# Función propia que devuelve un escalar por grupo
def grade_spread(group):
    return group['math'].max() - group['math'].min()

df.groupby('group').apply(grade_spread, include_groups=False)

group
A     5
B    14
C    12
dtype: int64

In [ ]:
# Función propia que devuelve un DataFrame (p. ej., el mejor alumno de cada grupo)
def top_student(group):
    return group.nlargest(1, 'math')

df.groupby('group').apply(top_student, include_groups=False)

,,student,math,physics,chemistry
group,,,,,
A,1,Bob,90,85,88
B,3,Diana,92,88,95
C,4,Eve,88,90,85


### Comparación: `agg`, `transform` y `apply` con groupby

| Método | Forma de la salida | Caso de uso |
|--------|--------------|----------|
| `agg` | Una fila por grupo | Estadísticos resumen, varias agregaciones |
| `transform` | Igual que la entrada | Llevar estadísticos de grupo de vuelta a las filas |
| `apply` | Flexible | Operaciones complejas, lógica propia |

## El método `assign`

`assign` crea columnas nuevas (o sobrescribe las existentes) y devuelve un DataFrame nuevo. Es especialmente útil para encadenar métodos.

In [ ]:
# Forma tradicional de añadir una columna
df_copy = df.copy()
df_copy['average'] = (df_copy['math'] + df_copy['physics'] + df_copy['chemistry']) / 3
df_copy

,student,group,math,physics,chemistry,average
0,Alice,A,85,80,75,80.000000
1,Bob,A,90,85,88,87.666667
2,Charlie,B,78,82,80,80.000000
3,Diana,B,92,88,95,91.666667
4,Eve,C,88,90,85,87.666667
5,Frank,C,76,70,72,72.666667


In [ ]:
# Con assign (no modifica el original)
df.assign(
    average=(df['math'] + df['physics'] + df['chemistry']) / 3
)

,student,group,math,physics,chemistry,average
0,Alice,A,85,80,75,80.000000
1,Bob,A,90,85,88,87.666667
2,Charlie,B,78,82,80,80.000000
3,Diana,B,92,88,95,91.666667
4,Eve,C,88,90,85,87.666667
5,Frank,C,76,70,72,72.666667


In [ ]:
# assign con lambda: hace referencia al DataFrame que se está transformando
df.assign(
    average=lambda x: (x['math'] + x['physics'] + x['chemistry']) / 3
)

,student,group,math,physics,chemistry,average
0,Alice,A,85,80,75,80.000000
1,Bob,A,90,85,88,87.666667
2,Charlie,B,78,82,80,80.000000
3,Diana,B,92,88,95,91.666667
4,Eve,C,88,90,85,87.666667
5,Frank,C,76,70,72,72.666667


In [ ]:
# Varias columnas a la vez, con dependencias entre ellas
df.assign(
    average=lambda x: (x['math'] + x['physics'] + x['chemistry']) / 3,
    passed=lambda x: x['average'] >= 80,  # ¡Usa la columna average que se acaba de crear!
    group_avg=lambda x: x.groupby('group')['average'].transform('mean')
)

,student,group,math,physics,chemistry,average,passed,group_avg
0,Alice,A,85,80,75,80.000000,True,83.833333
1,Bob,A,90,85,88,87.666667,True,83.833333
2,Charlie,B,78,82,80,80.000000,True,85.833333
3,Diana,B,92,88,95,91.666667,True,85.833333
4,Eve,C,88,90,85,87.666667,True,80.166667
5,Frank,C,76,70,72,72.666667,False,80.166667


**¿Por qué usar `assign`?**
- Permite encadenar métodos (estilo funcional)
- No modifica el DataFrame original
- Las funciones lambda pueden hacer referencia a columnas creadas en la misma llamada a `assign`

In [ ]:
# Ejemplo de encadenamiento de métodos
(
    df
    .assign(average=lambda x: x[['math', 'physics', 'chemistry']].mean(axis=1))
    .query('average >= 80')
    .sort_values('average', ascending=False)
)

,student,group,math,physics,chemistry,average
3,Diana,B,92,88,95,91.666667
4,Eve,C,88,90,85,87.666667
1,Bob,A,90,85,88,87.666667
0,Alice,A,85,80,75,80.000000
2,Charlie,B,78,82,80,80.000000


## Remodelar datos: `melt` y `pivot`

### Formato ancho frente a formato largo

Los datos se pueden organizar en dos formatos principales:
- **Formato ancho** (*wide*): cada variable tiene su propia columna (p. ej., `math`, `physics`, `chemistry` como columnas separadas)
- **Formato largo** (*long*): las variables se apilan en dos columnas: una para el nombre de la variable y otra para el valor

Análisis distintos necesitan formatos distintos. `melt` convierte de ancho a largo, y `pivot` de largo a ancho.

In [ ]:
# Nuestros datos están ahora en formato ancho
df

,student,group,math,physics,chemistry
0,Alice,A,85,80,75
1,Bob,A,90,85,88
2,Charlie,B,78,82,80
3,Diana,B,92,88,95
4,Eve,C,88,90,85
5,Frank,C,76,70,72


### `melt`: de ancho a largo

`melt` «despivota» un DataFrame de formato ancho a formato largo.

In [ ]:
# Convertimos a formato largo
df_long = df.melt(
    id_vars=['student', 'group'],      # Columnas que se mantienen como identificadores
    value_vars=['math', 'physics', 'chemistry'],  # Columnas que se despivotan
    var_name='subject',                # Nombre de la columna de variables
    value_name='grade'                 # Nombre de la columna de valores
)
df_long

,student,group,subject,grade
0,Alice,A,math,85
1,Bob,A,math,90
2,Charlie,B,math,78
3,Diana,B,math,92
4,Eve,C,math,88
5,Frank,C,math,76
6,Alice,A,physics,80
7,Bob,A,physics,85
8,Charlie,B,physics,82
9,Diana,B,physics,88


In [ ]:
# Si se omite value_vars, se funden todas las columnas que no están en id_vars
df.melt(id_vars=['student', 'group'], var_name='subject', value_name='grade')

,student,group,subject,grade
0,Alice,A,math,85
1,Bob,A,math,90
2,Charlie,B,math,78
3,Diana,B,math,92
4,Eve,C,math,88
5,Frank,C,math,76
6,Alice,A,physics,80
7,Bob,A,physics,85
8,Charlie,B,physics,82
9,Diana,B,physics,88


**¿Cuándo es útil el formato largo?**
- Para agregar por categorías (p. ej., la nota media por asignatura)
- Muchas bibliotecas de visualización prefieren el formato largo
- Es más fácil filtrar por categoría

In [ ]:
# Nota media por asignatura (más fácil en formato largo)
df_long.groupby('subject')['grade'].mean()

subject
chemistry    82.500000
math         84.833333
physics      82.500000
Name: grade, dtype: float64

In [ ]:
# Nota media por grupo y asignatura
df_long.groupby(['group', 'subject'])['grade'].mean()

group  subject  
A      chemistry    81.5
       math         87.5
       physics      82.5
B      chemistry    87.5
       math         85.0
       physics      85.0
C      chemistry    78.5
       math         82.0
       physics      80.0
Name: grade, dtype: float64

### `pivot` y `pivot_table`: de largo a ancho

`pivot` remodela los datos de formato largo a formato ancho. `pivot_table` es parecido, pero gestiona las entradas duplicadas agregándolas.

In [ ]:
# pivot: volvemos al formato ancho
df_wide = df_long.pivot(
    index=['student', 'group'],  # Columnas que se usan como identificadores de fila
    columns='subject',            # Columna cuyos valores se convierten en columnas nuevas
    values='grade'                # Valores con los que se rellenan las columnas nuevas
)
df_wide

,subject,chemistry,math,physics
student,group,,,
Alice,A,75,85,80
Bob,A,88,90,85
Charlie,B,80,78,82
Diana,B,95,92,88
Eve,C,85,88,90
Frank,C,72,76,70


In [ ]:
# Limpiamos el resultado
df_wide.reset_index().rename_axis(None, axis=1)

,student,group,chemistry,math,physics
0,Alice,A,75,85,80
1,Bob,A,88,90,85
2,Charlie,B,80,78,82
3,Diana,B,95,92,88
4,Eve,C,85,88,90
5,Frank,C,72,76,70


**`pivot` falla con entradas duplicadas:**

In [ ]:
# Si intentamos pivotar datos agrupados, falla porque hay varios valores por celda
try:
    df_long.pivot(index='group', columns='subject', values='grade')
except ValueError as e:
    print(f"Error: {e}")

Error: Index contains duplicate entries, cannot reshape


### `pivot_table`: gestiona los duplicados agregándolos

In [ ]:
# pivot_table agrega los duplicados (por defecto, con la media)
df_long.pivot_table(
    index='group',
    columns='subject',
    values='grade',
    aggfunc='mean'
)

subject,chemistry,math,physics
group,,,
A,81.5,87.5,82.5
B,87.5,85.0,85.0
C,78.5,82.0,80.0


In [ ]:
# Varias agregaciones
df_long.pivot_table(
    index='group',
    columns='subject',
    values='grade',
    aggfunc=['mean', 'std']
)

mean                      std                     
subject chemistry  math physics  chemistry      math    physics
group                                                          
A            81.5  87.5    82.5   9.192388  3.535534   3.535534
B            87.5  85.0    85.0  10.606602  9.899495   4.242641
C            78.5  82.0    80.0   9.192388  8.485281  14.142136

In [ ]:
# Añadimos márgenes (totales)
df_long.pivot_table(
    index='group',
    columns='subject',
    values='grade',
    aggfunc='mean',
    margins=True,
    margins_name='Overall'
)

subject,chemistry,math,physics,Overall
group,,,,
A,81.5,87.500000,82.5,83.833333
B,87.5,85.000000,85.0,85.833333
C,78.5,82.000000,80.0,80.166667
Overall,82.5,84.833333,82.5,83.277778


### `pivot_table` frente a `groupby` + `agg`

`pivot_table` es básicamente un `groupby` seguido de un remodelado. Estos dos son equivalentes:

In [ ]:
# Con pivot_table
pt = df_long.pivot_table(index='group', columns='subject', values='grade', aggfunc='mean')
pt

subject,chemistry,math,physics
group,,,
A,81.5,87.5,82.5
B,87.5,85.0,85.0
C,78.5,82.0,80.0


In [ ]:
# Equivalente con groupby
gb = df_long.groupby(['group', 'subject'])['grade'].mean().unstack()
gb

subject,chemistry,math,physics
group,,,
A,81.5,87.5,82.5
B,87.5,85.0,85.0
C,78.5,82.0,80.0
